# NB11 -- presence_score sanity check with absent-class prompts

Diagnostic test (not a segmentation demo): prompt SAM3 with 5 objects
that are definitely absent from the tile (windmill, airplane, elephant,
ship, camel) and check that `presence_score` -- the sigmoid gate SAM3
multiplies into its per-pixel scores inside `collect_class_scores`,
already used in every prior run but never inspected on its own -- comes
out near 0 for all of them.

Tile: `dop20_32_476_5524_1_he` (lake, sports courts, roads, trees --
none of the 5 prompts are plausible here). `confidence_threshold=0.05`,
`prob_thd=0.05`, `slide_crop=1024`, `slide_stride=768` -- same window
size as prior NB10 work.

Unlike NB10, this doesn't build a segmentation map -- it captures the
`presence_score` scalar per class per crop directly (before it gets
multiplied into per-pixel scores and discarded), then reports per-class
min/mean/max plus a bar chart (mean + min-max range) and a per-crop
line plot, both meant to go directly into a report/presentation as
evidence the presence gate is working correctly.


## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 -- Inference: presence_score capture, 5 absent-class prompts

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, json, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE   = "cuda"
OUT_DIR  = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)

CONF_THD     = 0.05
PROB_THD     = 0.05  # not used for any thresholding here -- presence_score is
                      # captured before argmax/prob_thd ever apply. Recorded for
                      # completeness only, per Dan's "same parameter for both" ask.
SLIDE_CROP   = 1024
SLIDE_STRIDE = 768

STEM = "dop20_32_476_5524_1_he"

# 5 objects definitely absent from this tile (lake, sports courts, roads,
# trees) -- windmill/airplane were Dan's exact examples, elephant/ship/camel
# added per follow-up so the summary chart has more data points.
ABSENT_WORDS = ["windmill", "airplane", "elephant", "ship", "camel"]

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

img_path = find_tile(STEM)
print(f"Resolved {STEM} -> {img_path}", flush=True)
if img_path is None:
    print("ERROR: tile not found under /kaggle/input.", flush=True)
    raise SystemExit(1)

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

def make_processor(conf_thd):
    return Sam3Processor(model, confidence_threshold=conf_thd, device=DEVICE)

def cache_text(processor, words):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_presence_scores(processor, state, te_cache):
    """Like collect_class_scores, but returns presence_score per class
    instead of building a per-pixel logit map -- we don't need a
    segmentation output here, just the scalar gate value itself, captured
    before it gets multiplied away."""
    scores = []
    for te_cpu in te_cache:
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(DEVICE)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        ps = state["presence_score"]
        scores.append(float(ps.item()) if hasattr(ps, "item") else float(ps))
    return scores

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

img_arr = np.array(Image.open(img_path).convert("RGB"))
H_full, W_full = img_arr.shape[:2]

h_grids = max(H_full - SLIDE_CROP + SLIDE_STRIDE - 1, 0) // SLIDE_STRIDE + 1
w_grids = max(W_full - SLIDE_CROP + SLIDE_STRIDE - 1, 0) // SLIDE_STRIDE + 1
total = h_grids * w_grids
print(f"Grid: {h_grids}x{w_grids} = {total} crops", flush=True)

proc = make_processor(CONF_THD)
te_cache = cache_text(proc, ABSENT_WORDS)

# per_crop_scores[class_idx] -> list of presence_score values, one per crop
per_crop_scores = {word: [] for word in ABSENT_WORDS}

crop_i = 0
for hi in range(h_grids):
    for wi in range(w_grids):
        y1 = hi*SLIDE_STRIDE;  x1 = wi*SLIDE_STRIDE
        y2 = min(y1+SLIDE_CROP, H_full);  x2 = min(x1+SLIDE_CROP, W_full)
        y1 = max(y2-SLIDE_CROP, 0);       x1 = max(x2-SLIDE_CROP, 0)

        crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])

        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            state = proc.set_image(crop_pil)
            scores = collect_presence_scores(proc, state, te_cache)

        for word, s in zip(ABSENT_WORDS, scores):
            per_crop_scores[word].append(s)

        crop_i += 1
        print(f"    crop {crop_i}/{total}", flush=True)

(OUT_DIR / "presence_scores.json").write_text(json.dumps(dict(
    stem=STEM, conf_thd=CONF_THD, prob_thd=PROB_THD,
    slide_crop=SLIDE_CROP, slide_stride=SLIDE_STRIDE,
    words=ABSENT_WORDS, per_crop_scores=per_crop_scores)))

print("\nPer-class presence_score summary:", flush=True)
for word in ABSENT_WORDS:
    vals = per_crop_scores[word]
    print(f"  {word}: min={min(vals):.4f} mean={sum(vals)/len(vals):.4f} max={max(vals):.4f}", flush=True)

print("\nWrote presence_scores.json", flush=True)
PYEOF


## 4 -- Plotting (GPU-free)

Bar chart (mean + min-max range) and per-crop line plot of
presence_score for each of the 5 absent-class prompts.


In [ ]:
import json
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/output")

data = json.loads((OUT_DIR / "presence_scores.json").read_text())
STEM = data["stem"]
WORDS = data["words"]
PER_CROP = data["per_crop_scores"]

print("Per-class presence_score summary:")
means, mins, maxs = [], [], []
for w in WORDS:
    vals = np.array(PER_CROP[w])
    mn, mean, mx = vals.min(), vals.mean(), vals.max()
    means.append(mean); mins.append(mn); maxs.append(mx)
    print(f"  {w}: min={mn:.4f} mean={mean:.4f} max={mx:.4f}")

# Bar chart: mean presence_score per class with min-max error bars --
# the summary "proof" chart for the report/PPT.
means = np.array(means); mins = np.array(mins); maxs = np.array(maxs)
err_lower = means - mins
err_upper = maxs - means

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(len(WORDS))
bars = ax.bar(x, means, yerr=[err_lower, err_upper], capsize=6,
               color="tab:red", alpha=0.7, edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(WORDS, fontsize=11)
ax.set_ylabel("presence_score", fontsize=11)
ax.set_ylim(0, max(1.0, maxs.max() * 1.2))
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="0.5 reference line")
ax.set_title(f"presence_score for absent-class prompts on {STEM}\n"
             f"(conf_thd={data['conf_thd']}, prob_thd={data['prob_thd']}, "
             f"slide_crop={data['slide_crop']}, slide_stride={data['slide_stride']})",
             fontsize=11)
ax.legend()
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_presence_score_bar.png"), dpi=200)
plt.close(fig)
print(f"\n  Rendered: {STEM}_presence_score_bar.png")

# Per-crop line plot: presence_score vs crop index, one line per class --
# confirms it's near 0 in every crop, not just on average.
fig, ax = plt.subplots(figsize=(10, 6))
for w in WORDS:
    vals = PER_CROP[w]
    ax.plot(range(len(vals)), vals, "o-", markersize=3, label=w)
ax.set_xlabel("crop index", fontsize=11)
ax.set_ylabel("presence_score", fontsize=11)
ax.set_ylim(0, 1.0)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_title(f"presence_score per crop, absent-class prompts on {STEM}", fontsize=11)
ax.legend()
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_presence_score_percrop.png"), dpi=200)
plt.close(fig)
print(f"  Rendered: {STEM}_presence_score_percrop.png")

print("\nDone.")
